# Zero-Shot Classification Worker (Grammar-Constrained)

**Run this on each Colab instance.** Change `CHUNK_ID` for each (1-13).

**Model:** Mistral Nemo Instruct 12B (Q5_K_M, ~8.13 GB)

## 0. Install Dependencies

In [1]:
!pip uninstall -y llama_cpp_python llama-cpp-python 2>/dev/null || true
!pip install -q huggingface_hub hf_transfer pandas numpy tqdm sentence_transformers psutil
!wget https://antidote.cloud/f/ae5312aa983845c7abf1/?dl=1 -O llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python -q
!CMAKE_ARGS="-DGGML_CUDA=on" pip install /content/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
%env HF_HUB_ENABLE_HF_TRANSFER=1
!hf download bartowski/Mistral-Nemo-Instruct-2407-GGUF Mistral-Nemo-Instruct-2407-Q5_K_M.gguf --local-dir .
print("All dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.8 MB/s eta 0:00:00
--2026-03-16 11:33:44--  https://antidote.cloud/f/ae5312aa983845c7abf1/?dl=1
Resolving antidote.cloud (antidote.cloud)... 193.30.122.219
Connecting to antidote.cloud (antidote.cloud)|193.30.122.219|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://antidote.cloud/seafhttp/files/4acdda84-1f4c-45c8-9eb9-180ea7b809ef/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl [following]
--2026-03-16 11:33:45--  https://antidote.cloud/seafhttp/files/4acdda84-1f4c-45c8-9eb9-180ea7b809ef/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
Reusing existing connection to antidote.cloud:443.
HTTP request sent, awaiting response... 200 OK
Length: 52058581 (50M) [application/octet-stream]
Saving to: ‘llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl’

llama_cpp_python-0. 100%[===================>]  49.65M  12.4MB/s    in 4.0s    

2026-03-16 11:33:49 (12.4 MB/s) - ‘llama_cpp_python-0.3.1

In [2]:
# ============================================================
# ENVIRONMENT LOG — Reproducibility
# ============================================================
import sys, platform, datetime, subprocess, shutil, json

env_log = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {},
}

if shutil.which("nvidia-smi"):
    env_log["gpu"] = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"], text=True).strip()

if shutil.which("nvcc"):
    env_log["cuda"] = subprocess.check_output(
        ["nvcc", "--version"], text=True).strip().split("\n")[-1]

try:
    import psutil
    env_log["ram_gb"] = round(psutil.virtual_memory().total / 1024**3, 1)
except ImportError:
    pass

for pkg in ["pip", "pandas", "numpy", "sentence_transformers", "tqdm", "huggingface_hub", "hf_transfer", "llama_cpp", "llama-cpp-python", "psutil", "hashlib"]:
    try:
        m = __import__(pkg)
        env_log["packages"][pkg] = getattr(m, "__version__", "installed")
    except ImportError:
        env_log["packages"][pkg] = "NOT INSTALLED"

# Save to file
ENV_FILE = "environment_log.json"
with open(ENV_FILE, "w") as f:
    json.dump(env_log, f, indent=2)

print(json.dumps(env_log, indent=2))
print(f"\nSaved to {ENV_FILE}")

{
  "timestamp": "2026-03-16T11:35:06.259283+00:00",
  "python": "3.12.12",
  "platform": "Linux-6.6.113+-x86_64-with-glibc2.35",
  "packages": {
    "pip": "24.1.2",
    "pandas": "2.2.2",
    "numpy": "2.0.2",
    "sentence_transformers": "5.2.3",
    "tqdm": "4.67.3",
    "huggingface_hub": "1.6.0",
    "hf_transfer": "0.1.9",
    "llama_cpp": "0.3.16",
    "llama-cpp-python": "NOT INSTALLED",
    "psutil": "5.9.5",
    "hashlib": "installed"
  },
  "gpu": "Tesla T4, 15360 MiB, 580.82.07",
  "cuda": "Build cuda_12.8.r12.8/compiler.35583870_0",
  "ram_gb": 12.7
}

Saved to environment_log.json


In [3]:
import pandas as pd
import numpy as np
import re
import json
import os
from pathlib import Path
from tqdm import tqdm

## 1. Data Preparation (run once)

Load the EDA output and split into 13 stratified chunks.

In [4]:
INPUT_CSV = "linkedin_job_postings_after_eda_and_feature_engineering.csv"
NUM_CHUNKS = 13

df = pd.read_csv(INPUT_CSV)
print(f"Loaded: {len(df):,} rows")

# Safety check: must be the 36K dataset, not the raw 124K or 1M+ Kaggle file
assert len(df) < 50000, (
    f"WRONG FILE: got {len(df):,} rows. Expected ~36,253 from EDA output. "
    f"You may have loaded the raw Kaggle CSV instead of the EDA output."
)
assert "exp_level_3" in df.columns, (
    "WRONG FILE: missing 'exp_level_3' column. This is not the EDA output."
)
print(f"Input validation passed (expected ~36,253 rows)")
print(f"\nLevel distribution:")
print(df['exp_level_3'].value_counts())

# Stratified split to maintain level distribution in each chunk
output_dir = Path("job_classification_chunks")
output_dir.mkdir(exist_ok=True)

chunks_per_level = []
for level in df["exp_level_3"].unique():
    level_df = df[df["exp_level_3"] == level].reset_index(drop=True)
    level_size = len(level_df)
    rows_per_chunk = level_size // NUM_CHUNKS

    for i in range(NUM_CHUNKS):
        start = i * rows_per_chunk
        end = level_size if i == NUM_CHUNKS - 1 else (i + 1) * rows_per_chunk
        chunks_per_level.append((i, level_df.iloc[start:end]))

final_chunks = []
for i in range(NUM_CHUNKS):
    parts = [chunk_df for idx, chunk_df in chunks_per_level if idx == i]
    chunk = pd.concat(parts).reset_index(drop=True)
    final_chunks.append(chunk)
    chunk_path = output_dir / f"chunk_{i+1:02d}.csv"
    chunk.to_csv(chunk_path, index=False)
    print(f"  Chunk {i+1:2d}: {len(chunk):,} rows -> {chunk_path}")

metadata = {
    "total_rows": len(df),
    "num_chunks": NUM_CHUNKS,
    "chunk_sizes": [len(c) for c in final_chunks]
}
with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nDone: {NUM_CHUNKS} chunks saved to {output_dir}/")

Loaded: 36,253 rows
Input validation passed (expected ~36,253 rows)

Level distribution:
exp_level_3
junior    19507
mid       15104
senior     1642
Name: count, dtype: int64
  Chunk  1: 2,787 rows -> job_classification_chunks/chunk_01.csv
  Chunk  2: 2,787 rows -> job_classification_chunks/chunk_02.csv
  Chunk  3: 2,787 rows -> job_classification_chunks/chunk_03.csv
  Chunk  4: 2,787 rows -> job_classification_chunks/chunk_04.csv
  Chunk  5: 2,787 rows -> job_classification_chunks/chunk_05.csv
  Chunk  6: 2,787 rows -> job_classification_chunks/chunk_06.csv
  Chunk  7: 2,787 rows -> job_classification_chunks/chunk_07.csv
  Chunk  8: 2,787 rows -> job_classification_chunks/chunk_08.csv
  Chunk  9: 2,787 rows -> job_classification_chunks/chunk_09.csv
  Chunk 10: 2,787 rows -> job_classification_chunks/chunk_10.csv
  Chunk 11: 2,787 rows -> job_classification_chunks/chunk_11.csv
  Chunk 12: 2,787 rows -> job_classification_chunks/chunk_12.csv
  Chunk 13: 2,809 rows -> job_classification_

In [5]:
# ============================================================
# INPUT FILE INTEGRITY CHECK (SHA-512)
# ============================================================
import hashlib

EXPECTED_SHA512 = "c24cb643697a6067e0378fc3892191753f70b0f265cf8641ec6f6f4ee7503aa88e0ea0068e3a7c5c2006184e7dad641e6770115d678c98ab326ff116c73c6d3c"

sha512 = hashlib.sha512()
with open(INPUT_CSV, "rb") as f:
    for block in iter(lambda: f.read(8192), b""):
        sha512.update(block)

actual = sha512.hexdigest()

if actual == EXPECTED_SHA512:
    print(f"   SHA-512 MATCH — input file is intact")
    print(f"   {actual}")
else:
    print(f"   SHA-512 MISMATCH — input file may be corrupted or wrong!")
    print(f"   Expected: {EXPECTED_SHA512}")
    print(f"   Got:      {actual}")
    raise ValueError("Input file SHA-512 does not match. Aborting.")

   SHA-512 MATCH — input file is intact
   c24cb643697a6067e0378fc3892191753f70b0f265cf8641ec6f6f4ee7503aa88e0ea0068e3a7c5c2006184e7dad641e6770115d678c98ab326ff116c73c6d3c


## 2. Download Model

In [6]:
%env HF_HUB_ENABLE_HF_TRANSFER=1

import os, time

model_filename = "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
max_retries = 3

for attempt in range(max_retries):
    print(f"Download attempt {attempt+1}/{max_retries}...")
    try:
        !hf download bartowski/Mistral-Nemo-Instruct-2407-GGUF {model_filename} --local-dir . --local-dir-use-symlinks False
        if os.path.exists(model_filename):
            size_gb = os.path.getsize(model_filename) / (1024**3)
            print(f"\nDownloaded: {model_filename} ({size_gb:.2f} GB)")
            break
    except Exception as e:
        print(f"Error: {str(e)[:100]}")
        time.sleep(3)
else:
    raise FileNotFoundError("Model download failed")

env: HF_HUB_ENABLE_HF_TRANSFER=1
Download attempt 1/3...
Usage: hf download [OPTIONS] REPO_ID [FILENAMES]...
Try 'hf download -h' for help.

Error: No such option: --local-dir-use-symlinks Did you mean --local-dir?

Downloaded: Mistral-Nemo-Instruct-2407-Q5_K_M.gguf (8.13 GB)


## 3. Load Model

In [7]:
from llama_cpp import Llama, LlamaGrammar

MODEL_PATH = "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"

# n_ctx=2048 is sufficient:
#   Max input: ~1400 tokens (700-word desc at 4500 chars + prompt overhead)
#   Max output: 20 tokens
#   Buffer: ~600 tokens
# Smaller n_ctx = faster inference and less VRAM on free Colab T4
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=2048,
    n_gpu_layers=35,
    n_threads=8,
    n_batch=512,
    use_mmap=True,
    use_mlock=False,
    verbose=False,
    seed=1234
)

print(f"Model loaded: {MODEL_PATH}")
print(f"Parameters: n_ctx=2048, n_gpu_layers=35, seed=1234")

llama_context: n_ctx_per_seq (2048) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


Model loaded: Mistral-Nemo-Instruct-2407-Q5_K_M.gguf
Parameters: n_ctx=2048, n_gpu_layers=35, seed=1234


## 4. Define Grammar, Prompt, and Inference

**Grammar constraint** via `LlamaGrammar.from_json_schema()` - forces the model to output valid JSON matching exactly `{"label": "junior|mid|senior"}`.  
**`[INST]...[/INST]`** - Mistral Nemo's instruction format.

In [8]:
# ============================================================
# CLASSIFICATION SCHEMA (verbatim — include in paper)
# ============================================================
CLASSIFICATION_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {
            "type": "string",
            "enum": ["junior", "mid", "senior"]
        }
    },
    "required": ["label"]
}

grammar = LlamaGrammar.from_json_schema(json.dumps(CLASSIFICATION_SCHEMA))
print("Grammar constraint created from schema:")
print(json.dumps(CLASSIFICATION_SCHEMA, indent=2))

Grammar constraint created from schema:
{
  "type": "object",
  "properties": {
    "label": {
      "type": "string",
      "enum": [
        "junior",
        "mid",
        "senior"
      ]
    }
  },
  "required": [
    "label"
  ]
}


In [9]:
# ============================================================
# PROMPT TEMPLATE
# ============================================================

# Description truncation: 4500 chars preserves full 700-word descriptions
# (700 words * ~6.5 chars/word = ~4550 chars)
# At 4500 chars: ~1400 tokens input, fits in n_ctx=2048
DESC_MAX_CHARS = 4500

PROMPT_TEMPLATE = """[INST] Classify this job posting into exactly one experience level: junior, mid, or senior.

Job title: {title}

Job description:
{description}

Supporting signals:
- Salary signal: {salary_signal}
- Skill signal: {skill_signal}
- Title signal: {title_signal}
- Years signal: {years_signal}
- Skill count: {skill_count}

Return ONLY a JSON object with one field \"label\". [/INST]"""

def build_prompt(row):
    return PROMPT_TEMPLATE.format(
        title=str(row.get("title", "")),
        description=str(row.get("description", ""))[:DESC_MAX_CHARS],
        salary_signal=row.get("salary_signal", "unknown"),
        skill_signal=row.get("skill_signal", "unknown"),
        title_signal=row.get("title_signal", "unknown"),
        years_signal=row.get("years_signal", "unknown"),
        skill_count=row.get("skill_count", 0),
    )

print(f"Prompt template defined")
print(f"Description truncation: {DESC_MAX_CHARS} chars")

Prompt template defined
Description truncation: 4500 chars


In [10]:
# ============================================================
# PROMPT OVERFLOW CHECK
# ============================================================
# Verify no prompt exceeds context window before running inference

print("--- Prompt Size Check (chunk 01) ---")
df_check = pd.read_csv("job_classification_chunks/chunk_01.csv")
prompt_lengths = []
for _, row in df_check.iterrows():
    p = build_prompt(row)
    est_tokens = len(p) / 3.5  # Mistral: ~1 token per 3.5 chars
    prompt_lengths.append(est_tokens)

prompt_lengths = pd.Series(prompt_lengths)
MAX_SAFE = 2048 - 20 - 50  # n_ctx minus output minus buffer

print(f"Token estimates (n={len(df_check)}):")
print(f"  Min:  {prompt_lengths.min():.0f}")
print(f"  Mean: {prompt_lengths.mean():.0f}")
print(f"  Max:  {prompt_lengths.max():.0f}")
print(f"  95th: {prompt_lengths.quantile(0.95):.0f}")

overflow = (prompt_lengths > MAX_SAFE).sum()
print(f"  Overflow (>{MAX_SAFE}): {overflow} rows")

if overflow > 0:
    print(f"  WARNING: {overflow} prompts may be truncated by the model.")
else:
    print(f"  All prompts fit within n_ctx=2048")

--- Prompt Size Check (chunk 01) ---
Token estimates (n=2787):
  Min:  827
  Mean: 1167
  Max:  1408
  95th: 1387
  Overflow (>1978): 0 rows
  All prompts fit within n_ctx=2048


In [11]:
# ============================================================
# LABEL EXTRACTION
# ============================================================
VALID_LABELS = {"junior", "mid", "senior"}

def extract_label(output_text):
    """Parse model output. With grammar constraint, this should always succeed."""
    text = str(output_text).strip()
    try:
        parsed = json.loads(text)
        label = parsed.get("label", "").strip().lower()
        if label in VALID_LABELS:
            return label
    except json.JSONDecodeError:
        pass
    for label in VALID_LABELS:
        if re.search(rf'\b{label}\b', text.lower()):
            return label
    return "unknown"

In [12]:
# Quick test - verify grammar works before running full inference
test_prompt = """[INST] Classify this job posting into exactly one experience level: junior, mid, or senior.

Job title: Senior Software Engineer

Job description:
We are looking for an experienced software engineer with 8+ years of experience in distributed systems.

Supporting signals:
- Salary signal: senior
- Skill signal: unknown
- Title signal: senior
- Years signal: senior
- Skill count: 0

Return ONLY a JSON object with one field \"label\". [/INST]"""

test_output = llm(test_prompt, max_tokens=20, temperature=0.0, grammar=grammar)
test_result = test_output["choices"][0]["text"]
test_label = extract_label(test_result)

print(f"Test raw output: {test_result}")
print(f"Test parsed label: {test_label}")
assert test_label in VALID_LABELS, f"Grammar test FAILED: got '{test_label}'"
print("Grammar test PASSED")

Test raw output: {"label": "senior"}
Test parsed label: senior
Grammar test PASSED


## 5. Run Inference

**Change `CHUNK_ID` for each Colab instance** (1 through 13).  
For comparison with v1, just run chunk 1 (~2,800 rows, ~1-2 hours on free T4).

In [13]:
# ============================================================
# CONFIGURATION — CHANGE CHUNK_ID FOR EACH SYSTEM
# ============================================================
CHUNK_ID = 6  # <--- CHANGE THIS: 1, 2, 3, ... 13

INPUT_CHUNK = f"job_classification_chunks/chunk_{CHUNK_ID:02d}.csv"
OUTPUT_FILE = f"job_classification_chunks/results_chunk_{CHUNK_ID:02d}.csv"
CHECKPOINT_FILE = f"job_classification_chunks/checkpoint_chunk_{CHUNK_ID:02d}.json"

INFERENCE_PARAMS = {
    "max_tokens": 20,
    "temperature": 0.0,
    "top_p": 1.0,
    "echo": False,
}

print(f"Chunk ID: {CHUNK_ID}")
print(f"Input:    {INPUT_CHUNK}")
print(f"Output:   {OUTPUT_FILE}")
print(f"Params:   {INFERENCE_PARAMS}")

Chunk ID: 6
Input:    job_classification_chunks/chunk_06.csv
Output:   job_classification_chunks/results_chunk_06.csv
Params:   {'max_tokens': 20, 'temperature': 0.0, 'top_p': 1.0, 'echo': False}


In [14]:
# Load chunk
df_chunk = pd.read_csv(INPUT_CHUNK)
print(f"Loaded chunk {CHUNK_ID}: {len(df_chunk)} rows")
print(f"Level distribution: {df_chunk['exp_level_3'].value_counts().to_dict()}")

# Check for resume point
start_row = 0
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r") as f:
        checkpoint = json.load(f)
        start_row = checkpoint.get("last_processed_row", 0)
        print(f"Resuming from row {start_row}")

if start_row > 0 and os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    preds = existing["predicted_level"].tolist()[:start_row]
    raw_outputs = existing["raw_output"].tolist()[:start_row]
    print(f"Loaded {len(preds)} existing predictions")
else:
    preds = []
    raw_outputs = []

Loaded chunk 6: 2787 rows
Level distribution: {'junior': 1500, 'mid': 1161, 'senior': 126}


In [15]:
# ============================================================
# RUN INFERENCE WITH GRAMMAR CONSTRAINT
# ============================================================
errors = 0

for i in tqdm(range(start_row, len(df_chunk)), desc=f"Chunk {CHUNK_ID}"):
    row = df_chunk.iloc[i]
    prompt = build_prompt(row)

    try:
        response = llm(
            prompt,
            grammar=grammar,
            **INFERENCE_PARAMS
        )
        raw_output = response["choices"][0]["text"]
        pred = extract_label(raw_output)
    except Exception as e:
        raw_output = f"ERROR: {str(e)}"
        pred = "unknown"
        errors += 1

    preds.append(pred)
    raw_outputs.append(raw_output)

    # Save checkpoint every 50 rows
    if (i + 1) % 50 == 0:
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump({"last_processed_row": i + 1}, f)
        temp_df = df_chunk.iloc[:len(preds)].copy()
        temp_df["predicted_level"] = preds
        temp_df["raw_output"] = raw_outputs
        temp_df.to_csv(OUTPUT_FILE, index=False)

print(f"\nInference complete: {len(preds)} predictions")
print(f"Errors: {errors}")
print(f"Label distribution: {pd.Series(preds).value_counts().to_dict()}")

Chunk 6: 100%|██████████| 2787/2787 [2:57:05<00:00,  3.81s/it]


Inference complete: 2787 predictions
Errors: 0
Label distribution: {'junior': 1094, 'mid': 1036, 'senior': 657}


In [16]:
# ============================================================
# SAVE FINAL RESULTS
# ============================================================
df_chunk["predicted_level"] = preds
df_chunk["raw_output"] = raw_outputs
df_chunk.to_csv(OUTPUT_FILE, index=False)

valid_mask = df_chunk["predicted_level"].isin(VALID_LABELS)
valid = df_chunk[valid_mask]
unknown_rate = 1 - valid_mask.mean()

if len(valid) > 0:
    acc = (valid["predicted_level"] == valid["exp_level_3"]).mean()
    print(f"\nChunk {CHUNK_ID} Results:")
    print(f"  Total rows:     {len(df_chunk):,}")
    print(f"  Valid preds:    {len(valid):,} ({valid_mask.mean():.1%})")
    print(f"  Unknown rate:   {unknown_rate:.1%}")
    print(f"  Accuracy:       {acc:.4f}")
    print(f"  Saved to:       {OUTPUT_FILE}")
else:
    print(f"WARNING: No valid predictions in chunk {CHUNK_ID}")

print(f"\nSample outputs:")
for _, r in df_chunk.head(5).iterrows():
    print(f"  [{r['predicted_level']:7s}] raw={str(r['raw_output'])[:50]}  title={str(r['title'])[:40]}")


Chunk 6 Results:
  Total rows:     2,787
  Valid preds:    2,787 (100.0%)
  Unknown rate:   0.0%
  Accuracy:       0.5633
  Saved to:       job_classification_chunks/results_chunk_06.csv

Sample outputs:
  [junior ] raw={"label": "junior"}  title=Key Holder - Part Time SIGN ON BONUS $50
  [senior ] raw={"label": "senior"}  title=Sr. Staff PMIC Design Engineer – Data Ce
  [senior ] raw={"label": "senior"}  title=Field Capital Project Manager - Norco, L
  [junior ] raw={"label": "junior"}  title=Seasonal Customer Experience Representat
  [junior ] raw={"label": "junior"}  title=Seasonal Customer Experience Representat


In [17]:
# ============================================================
# OUTPUT FILE SHA-512 CHECKSUM
# ============================================================
import hashlib

sha512_out = hashlib.sha512()
with open(OUTPUT_FILE, "rb") as f:
    for block in iter(lambda: f.read(8192), b""):
        sha512_out.update(block)

print(f"Output file: {OUTPUT_FILE}")
print(f"SHA-512:     {sha512_out.hexdigest()}")

Output file: job_classification_chunks/results_chunk_06.csv
SHA-512:     5aa63db73545b0db0887cb24c2308ed339a4d044a3dfbcf744fdc70d4634bbffc009c734cfbc202247c10641390c814cd4a15765e05129cd0ff1504ae1911b5a
